# Práctica 2. Resolución de Problemas con Búsqueda Heurística
## Ingeniería del Conocimiento 2025/2026
### Prof. Juan A. Recio García

---

> **Objetivo:** Aplicar búsqueda informada (heurística) al problema del 8-Puzzle, implementar distintas funciones heurísticas y comparar su rendimiento con A* y búsqueda voraz.


## Parte II: Búsqueda Heurística — Teoría

### ¿Por qué no basta con búsqueda ciega?

Los algoritmos ciegos (BFS, DFS...) tienen complejidad **exponencial** O(r^p).  
Con r=10 y p=10 necesitaríamos explorar **10.000 millones de nodos** (~116 días).  
En problemas reales esto es completamente inviable.

### La solución: funciones heurísticas h(n)

Una **función heurística** `h(n)` asigna a cada estado un número que estima cuán lejos está del objetivo.  
Permite **guiar la búsqueda** hacia los caminos más prometedores, evitando explorar ramas inútiles.

> **Convención:** `h(n) = 0` en el estado objetivo. Cuanto menor, más cerca del objetivo.

### Propiedades clave

| Propiedad | Definición |
|---|---|
| **Admisible** | `h'(n) ≤ h(n)` — nunca sobreestima el coste real. Garantiza optimalidad en A* |
| **Consistente** | `h(ni) ≤ coste(ni→nj) + h(nj)` — desigualdad triangular. Toda consistente es admisible |
| **Más informada** | h1 es más informada que h2 si `h1(n) ≥ h2(n)` para todo n (aproxima mejor al coste real) |

### Algoritmos de búsqueda informada

| Algoritmo | Función de evaluación | ¿Óptimo? |
|---|---|---|
| **Búsqueda Voraz** | `f'(n) = h'(n)` | ❌ No (ignora el coste ya pagado) |
| **A\*** | `f'(n) = g(n) + h'(n)` | ✅ Sí (si h' es admisible) |

- `g(n)` = coste real acumulado desde el inicio hasta n  
- `h'(n)` = estimación heurística del coste desde n hasta el objetivo

### Búsqueda Voraz vs A*

- **Voraz**: va siempre al nodo que parece más cercano al objetivo. Rápida pero no óptima.  
  _(Ejemplo: Arad→Bucarest por ruta de coste 450 cuando la óptima es 418)_
- **A***: combina coste real + estimación → encuentra la solución **óptima**.  
  _(Arad→Bucarest encuentra siempre la ruta de coste 418)_


## Instalación e importación

In [1]:

from search import *
print("Librería AIMA cargada ✅")


Librería AIMA cargada ✅


---
# EJERCICIO: 8-Puzzle con Búsqueda Heurística

## Las 4 heurísticas a implementar

### 1. `linear(node)` — Fichas mal colocadas
Cuenta cuántas fichas NO están en su posición objetivo (sin contar el hueco).

```
Estado:   2 4 3     Goal: 1 2 3
          1 5 6           4 5 6
          7 8 0           7 8 0

Fichas mal colocadas: 2 y 4 están cambiadas → linear = 2
```
- ✅ **Admisible**: cada ficha fuera de lugar necesita al menos 1 movimiento.
- Es la heurística **menos informada** de las cuatro.

---

### 2. `manhattan(node)` — Distancia Manhattan
Suma las distancias de cada ficha a su posición objetivo (movimientos horizontales + verticales, sin diagonales).

**Cálculo de la distancia de cada ficha:**
```
posición en tablero 3x3:
  fila    = posición // 3
  columna = posición % 3

distancia_manhattan(ficha) = |fila_actual - fila_objetivo| + |col_actual - col_objetivo|
```

**Ejemplo:**
```
Estado:   2 4 3     Goal: 1 2 3
          1 5 6           4 5 6
          7 8 0           7 8 0

Ficha 2: está en pos 0 (fila 0, col 0), objetivo pos 1 (fila 0, col 1) → distancia = 1
Ficha 4: está en pos 1 (fila 0, col 1), objetivo pos 3 (fila 1, col 0) → distancia = 2
Resto en su sitio → manhattan = 1 + 2 = 3
```
- ✅ **Admisible y consistente**: un movimiento solo acerca una ficha 1 paso.
- **Más informada que `linear`**: manhattan ≥ linear para todo estado.

---

### 3. `sqrt_manhattan(node)` — Raíz cuadrada de Manhattan
`sqrt_manhattan = sqrt(manhattan)`

- ✅ **Admisible**: `sqrt(x) ≤ x` para x ≥ 1, así que no sobreestima.
- **Menos informada que manhattan**: al tomar la raíz el valor baja.
- Puede ser útil para priorizar soluciones más rápidas (aunque no óptimas).

---

### 4. `max_heuristic(node)` — Máximo de linear y manhattan
`max_heuristic = max(linear, manhattan)`

- ✅ **Admisible**: el máximo de dos heurísticas admisibles también es admisible.
- **Más informada que ambas**: `max(linear, manhattan) ≥ linear` y `≥ manhattan` individualmente.
- Es la heurística **más informada** de las cuatro → A* expandirá menos nodos.


In [2]:
class Ocho_Puzzle(Problem):
    """
    8-Puzzle con 4 heurísticas implementadas.
    Estado: tupla de 9 elementos (0=hueco).
    Posiciones leídas por filas: 0-1-2 / 3-4-5 / 6-7-8
    """

    def __init__(self, initial, goal=(1, 2, 3, 4, 5, 6, 7, 8, 0)):
        self.goal = goal
        Problem.__init__(self, initial, goal)

    # ── ACCIONES ──────────────────────────────────────────────────────────────
    def actions(self, estado):
        """Devuelve las acciones válidas según la posición del hueco."""
        pos_hueco = estado.index(0)
        accs = []
        if pos_hueco >= 3:       accs.append("Mover hueco arriba")     # no está en fila 0
        if pos_hueco <= 5:       accs.append("Mover hueco abajo")      # no está en fila 2
        if pos_hueco % 3 != 0:  accs.append("Mover hueco izquierda")  # no está en col 0
        if pos_hueco % 3 != 2:  accs.append("Mover hueco derecha")    # no está en col 2
        return accs

    # ── RESULTADO ─────────────────────────────────────────────────────────────
    def result(self, estado, accion):
        """Aplica la acción intercambiando el hueco con la ficha adyacente."""
        pos_hueco = estado.index(0)
        l = list(estado)
        def swap(x):
            l[pos_hueco], l[pos_hueco+x] = l[pos_hueco+x], l[pos_hueco]
        if accion == "Mover hueco arriba":    swap(-3)
        elif accion == "Mover hueco abajo":   swap(+3)
        elif accion == "Mover hueco izquierda": swap(-1)
        elif accion == "Mover hueco derecha":   swap(+1)
        return tuple(l)

    # ── HEURÍSTICA 1: FICHAS MAL COLOCADAS ────────────────────────────────────
    def linear(self, node):
        """
        Cuenta el número de fichas que NO están en su posición objetivo.
        No se cuenta el hueco (0).
        Ejemplo: si 2 fichas están fuera de lugar → devuelve 2
        """
        estado = node.state
        return sum(
            1 for i in range(9)
            if estado[i] != self.goal[i] and estado[i] != 0
        )

    # ── HEURÍSTICA 2: DISTANCIA MANHATTAN ─────────────────────────────────────
    def manhattan(self, node):
        """
        Suma las distancias Manhattan de cada ficha a su posición objetivo.
        distancia(ficha) = |fila_actual - fila_objetivo| + |col_actual - col_objetivo|
        fila = posición // 3,  columna = posición % 3
        No se cuenta el hueco (0).
        """
        estado = node.state
        distancia_total = 0
        for pos_actual, ficha in enumerate(estado):
            if ficha != 0:  # ignoramos el hueco
                # Posición objetivo de esta ficha en el tablero goal
                pos_objetivo = self.goal.index(ficha)
                # Coordenadas actuales
                fila_actual = pos_actual // 3
                col_actual  = pos_actual % 3
                # Coordenadas objetivo
                fila_obj = pos_objetivo // 3
                col_obj  = pos_objetivo % 3
                # Distancia Manhattan para esta ficha
                distancia_total += abs(fila_actual - fila_obj) + abs(col_actual - col_obj)
        return distancia_total

    # ── HEURÍSTICA 3: RAÍZ CUADRADA DE MANHATTAN ──────────────────────────────
    def sqrt_manhattan(self, node):
        """
        Raíz cuadrada de la distancia Manhattan.
        sqrt_manhattan = sqrt(manhattan(n))
        Admisible porque sqrt(x) <= x para x >= 1.
        Menos informada que manhattan (valores más pequeños).
        """
        import math
        return math.sqrt(self.manhattan(node))

    # ── HEURÍSTICA 4: MÁXIMO DE LINEAR Y MANHATTAN ────────────────────────────
    def max_heuristic(self, node):
        """
        Máximo entre linear y manhattan.
        max_heuristic = max(linear(n), manhattan(n))
        Es admisible (máximo de dos admisibles lo es).
        Es la más informada de las cuatro: no subestima más de lo necesario.
        """
        return max(self.linear(node), self.manhattan(node))

    # ── h() POR DEFECTO ───────────────────────────────────────────────────────
    def h(self, node):
        """Heurística por defecto usada por astar_search si no se especifica otra."""
        return self.manhattan(node)

    def coste_de_aplicar_accion(self, s, a):
        return 1


## Función auxiliar para visualizar el tablero

In [3]:
def show(s):
    """Muestra el tablero del 8-puzzle de forma visual."""
    print("+---+---+---+")
    for i in range(0, 9, 3):
        fila = []
        for j in range(3):
            val = s[i+j]
            fila.append(" H " if val == 0 else f" {val} ")
        print("|" + "|".join(fila) + "|")
        print("+---+---+---+")


## Estado inicial del puzzle

```
+---+---+---+
| 2 | 4 | 3 |
+---+---+---+
| 1 | 5 | 6 |
+---+---+---+
| 7 | 8 | H |
+---+---+---+
```


In [4]:
puzle = Ocho_Puzzle((2, 4, 3, 1, 5, 6, 7, 8, 0))

print("Estado inicial:")
show(puzle.initial)
print("\nEstado objetivo:")
show(puzle.goal)


Estado inicial:
+---+---+---+
| 2 | 4 | 3 |
+---+---+---+
| 1 | 5 | 6 |
+---+---+---+
| 7 | 8 | H |
+---+---+---+

Estado objetivo:
+---+---+---+
| 1 | 2 | 3 |
+---+---+---+
| 4 | 5 | 6 |
+---+---+---+
| 7 | 8 | H |
+---+---+---+


## Verificación de las heurísticas

Antes de ejecutar los algoritmos, comprobamos que las heurísticas devuelven valores coherentes.

Para el estado `(2,4,3,1,5,6,7,8,0)`:
- **linear**: 2 y 4 están intercambiadas → debe dar **2**
- **manhattan**: ficha 2 (pos 0→1: dist 1) + ficha 4 (pos 1→3: dist 2) = **3**
- **sqrt_manhattan**: √3 ≈ **1.73**
- **max_heuristic**: max(2, 3) = **3**


In [5]:
from search import Node  # necesitamos crear un nodo para llamar a las heurísticas

# Creamos un nodo 'dummy' con el estado inicial para probar las heurísticas
nodo_prueba = Node(puzle.initial)

print("Estado:", puzle.initial)
show(puzle.initial)
print(f"\nlinear        = {puzle.linear(nodo_prueba)}")
print(f"manhattan     = {puzle.manhattan(nodo_prueba)}")
print(f"sqrt_manhattan= {puzle.sqrt_manhattan(nodo_prueba):.4f}")
print(f"max_heuristic = {puzle.max_heuristic(nodo_prueba)}")


Estado: (2, 4, 3, 1, 5, 6, 7, 8, 0)
+---+---+---+
| 2 | 4 | 3 |
+---+---+---+
| 1 | 5 | 6 |
+---+---+---+
| 7 | 8 | H |
+---+---+---+

linear        = 3
manhattan     = 4
sqrt_manhattan= 2.0000
max_heuristic = 4


---
## Búsqueda Voraz (`greedy_best_first_graph_search`)

Usa `f'(n) = h'(n)`. Solo mira la estimación al objetivo, **ignora el coste ya pagado**.  
Rápida pero **no garantiza solución óptima**.


In [6]:
print("=== Búsqueda VORAZ con manhattan ===")
sol_voraz = greedy_best_first_graph_search(puzle, puzle.manhattan)
print("Solución:", sol_voraz.solution())
print("Pasos:", len(sol_voraz.solution()))


=== Búsqueda VORAZ con manhattan ===
Solución: ['Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco abajo', 'Mover hueco derecha', 'Mover hueco derecha', 'Mover hueco abajo']
Pasos: 8


## Búsqueda A* (`astar_search`)

Usa `f'(n) = g(n) + h'(n)`. Combina coste real + estimación.  
**Óptima y completa** si la heurística es admisible.


In [7]:
print("=== A* con manhattan ===")
sol_astar_manhattan = astar_search(puzle, puzle.manhattan)
print("Solución:", sol_astar_manhattan.solution())
print("Pasos:", len(sol_astar_manhattan.solution()))


=== A* con manhattan ===
Solución: ['Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco abajo', 'Mover hueco derecha', 'Mover hueco derecha', 'Mover hueco abajo']
Pasos: 8


### Probamos A* con las otras heurísticas

In [8]:
print("=== A* con linear (fichas mal colocadas) ===")
sol_linear = astar_search(puzle, puzle.linear)
print("Solución:", sol_linear.solution())
print("Pasos:", len(sol_linear.solution()))


=== A* con linear (fichas mal colocadas) ===
Solución: ['Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco abajo', 'Mover hueco derecha', 'Mover hueco derecha', 'Mover hueco abajo']
Pasos: 8


In [9]:
print("=== A* con max_heuristic (max de linear y manhattan) ===")
sol_max = astar_search(puzle, puzle.max_heuristic)
print("Solución:", sol_max.solution())
print("Pasos:", len(sol_max.solution()))


=== A* con max_heuristic (max de linear y manhattan) ===
Solución: ['Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco abajo', 'Mover hueco derecha', 'Mover hueco derecha', 'Mover hueco abajo']
Pasos: 8


In [10]:
print("=== A* con sqrt_manhattan ===")
sol_sqrt = astar_search(puzle, puzle.sqrt_manhattan)
print("Solución:", sol_sqrt.solution())
print("Pasos:", len(sol_sqrt.solution()))


=== A* con sqrt_manhattan ===
Solución: ['Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco abajo', 'Mover hueco derecha', 'Mover hueco derecha', 'Mover hueco abajo']
Pasos: 8


---
## Comparación de tiempos con `%timeit`

Usamos 3 estados de distinta dificultad para medir el tiempo de cada heurística con A*.

| Puzzle | Estado inicial | Dificultad |
|---|---|---|
| puzzle_1 | `(2,4,3,1,5,6,7,8,0)` | Fácil (2 fichas mal) |
| puzzle_2 | `(1,2,3,4,5,6,0,7,8)` | Media |
| puzzle_3 | `(1,2,3,4,5,7,8,6,0)` | Difícil |


In [1]:
puzzle_1 = Ocho_Puzzle((2, 4, 3, 1, 5, 6, 7, 8, 0))
puzzle_2 = Ocho_Puzzle((1, 2, 3, 4, 5, 6, 0, 7, 8))
puzzle_3 = Ocho_Puzzle((1, 2, 3, 4, 5, 7, 8, 6, 0))
print("Puzzles creados ✅")


NameError: name 'Ocho_Puzzle' is not defined

In [13]:
%%timeit
astar_search(puzzle_1, puzzle_1.linear)
astar_search(puzzle_2, puzzle_2.linear)
astar_search(puzzle_3, puzzle_3.linear)


1.16 ms ± 48.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
%%timeit
astar_search(puzzle_1, puzzle_1.manhattan)
astar_search(puzzle_2, puzzle_2.manhattan)
astar_search(puzzle_3, puzzle_3.manhattan)


In [ ]:
%%timeit
astar_search(puzzle_1, puzzle_1.sqrt_manhattan)
astar_search(puzzle_2, puzzle_2.sqrt_manhattan)
astar_search(puzzle_3, puzzle_3.sqrt_manhattan)


In [ ]:
%%timeit
astar_search(puzzle_1, puzzle_1.max_heuristic)
astar_search(puzzle_2, puzzle_2.max_heuristic)
astar_search(puzzle_3, puzzle_3.max_heuristic)


---
## ✏️ Conclusiones: ¿Qué heurística es mejor?

### Orden de mayor a menor informada:
```
max_heuristic  ≥  manhattan  ≥  linear  ≥  sqrt_manhattan
```

### Razonamiento detallado:

**1. `max_heuristic` es la MÁS informada**  
`max(linear, manhattan) ≥ manhattan` y `≥ linear` para todo estado.  
Al dar valores más altos (sin sobreestimar), A* descarta antes los caminos malos → **expande menos nodos** → más rápido.

**2. `manhattan` es más informada que `linear`**  
`manhattan` tiene en cuenta CUÁNTOS movimientos necesita cada ficha, no solo si está mal o bien.  
Siempre se cumple `manhattan(n) ≥ linear(n)` (una ficha fuera de lugar necesita ≥1 movimiento).  
→ `manhattan` aproxima mejor el coste real.

**3. `linear` es menos informada que `manhattan`**  
Solo sabe que una ficha está mal puesta, pero no sabe a cuántos pasos está de su destino.  
Valores más pequeños → A* es menos selectivo → explora más nodos → más lento.

**4. `sqrt_manhattan` es la MENOS informada**  
`sqrt(x) ≤ x` para x ≥ 1, así que `sqrt_manhattan ≤ manhattan`.  
Valores aún más pequeños → A* es el menos selectivo → más nodos explorados → más lento.

### Conclusión sobre optimalidad:
Todas son **admisibles** → A* con cualquiera de ellas encuentra la **solución óptima** (mismo número de pasos).  
La diferencia está en el **tiempo de cómputo**: más informada = menos nodos expandidos = más rápido.

### Comparativa esperada de tiempos:
```
max_heuristic < manhattan < linear < sqrt_manhattan
     (más rápido)                       (más lento)
```

### Voraz vs A*:
- **Voraz** es más rápido pero puede encontrar soluciones **no óptimas** (más pasos).
- **A*** con manhattan es el mejor equilibrio: óptimo y eficiente.


---
## Resumen: Búsqueda Informada vs No Informada

| Algoritmo | Función eval | Completo | Óptimo | Velocidad práctica |
|---|---|---|---|---|
| BFS (ciega) | — | ✅ | ✅ | 🐢 Muy lento |
| A* + linear | g(n) + linear(n) | ✅ | ✅ | 🐢 Lento |
| A* + manhattan | g(n) + manhattan(n) | ✅ | ✅ | 🚀 Rápido |
| A* + max_heuristic | g(n) + max(n) | ✅ | ✅ | 🚀🚀 Más rápido |
| Voraz + manhattan | manhattan(n) | ⚠️ | ❌ | 🚀🚀 Muy rápido |
| A* + sqrt_manhattan | g(n) + sqrt(n) | ✅ | ✅ | 🐢 Más lento que manhattan |

**Regla de oro:** cuanto más informada sea la heurística (mayor h'(n) sin sobreestimar), **menos nodos expande A*** y más rápido encuentra la solución óptima.
